# USGS NWIS Water Services — Streamflow & Groundwater

**What it is.** The USGS National Water Information System: real-time and historical
**streamflow**, **groundwater levels**, and water quality from tens of thousands of sites.

| | |
|---|---|
| **Coverage** | United States |
| **Cost / key** | **Free · no key** |
| **Docs** | https://waterservices.usgs.gov/ |

**Why it's in Tracera.** Surface- and ground-water availability is the backbone of irrigation
supply and drought context. Nearby stream discharge and well levels signal water stress.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Find streamflow gages near the field (bounding box)

In [2]:
def usgs_sites(param="00060", dtype="dv", half=0.4):
    """Find nearby sites for a parameter. 00060 = discharge, 72019 = groundwater depth."""
    bbox = f"{LON-half:.4f},{LAT-half*0.75:.4f},{LON+half:.4f},{LAT+half*0.75:.4f}"
    r = requests.get("https://waterservices.usgs.gov/nwis/site/", params={"format": "rdb",
        "bBox": bbox, "parameterCd": param, "siteStatus": "active", "hasDataTypeCd": dtype}, timeout=60)
    rows = [l for l in r.text.splitlines() if l and not l.startswith("#")]
    hdr = rows[0].split("\t")
    return pd.DataFrame([r.split("\t") for r in rows[2:]], columns=hdr)

gages = usgs_sites("00060", "dv")
print(f"{len(gages)} active streamflow gages near the field:")
gages[["site_no", "station_nm"]].head()

5 active streamflow gages near the field:


,site_no,station_nm
0,05451210,"South Fork Iowa River NE of New Providence, IA"
1,05470000,"South Skunk River near Ames, IA"
2,05470500,"Ioway Creek at Ames, IA"
3,05471000,"South Skunk River below Ioway Creek near Ames, IA"
4,05471200,"Indian Creek near Mingo, IA"


### 2. Core query — daily discharge time series for the nearest gage

In [3]:
site = gages.iloc[0]["site_no"]

def usgs_daily(site, param="00060", start="2023-05-01", end="2023-09-30"):
    r = requests.get("https://waterservices.usgs.gov/nwis/dv/", params={"format": "json",
        "sites": site, "parameterCd": param, "startDT": start, "endDT": end}, timeout=60)
    ts = r.json()["value"]["timeSeries"]
    vals = ts[0]["values"][0]["value"]
    df = pd.DataFrame(vals)
    df["dateTime"] = pd.to_datetime(df["dateTime"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.set_index("dateTime")["value"].rename("discharge_cfs")

flow = usgs_daily(site)
print(f"Site {site} — {gages.iloc[0]['station_nm']}")
print(f"Mean summer discharge: {flow.mean():.0f} cfs  (min {flow.min():.0f}, max {flow.max():.0f})")
flow.head()

Site 05451210 — South Fork Iowa River NE of New Providence, IA
Mean summer discharge: 65 cfs  (min 1, max 360)


dateTime
2023-05-01    96.7
2023-05-02    88.2
2023-05-03    81.8
2023-05-04    78.7
2023-05-05    78.1
Name: discharge_cfs, dtype: float64

### Notes & how to extend
- **Parameter codes:** `00060` discharge (cfs), `00065` gage height, `72019` groundwater depth
  below surface, `00010` water temperature, `00400` pH.
- **Real-time:** swap `/dv/` (daily) for `/iv/` (instantaneous) for live readings.
- **Groundwater:** call `usgs_sites("72019", "dv")` to find monitoring wells and pull levels —
  key for irrigation-well context.
- Correlate low-flow periods with the **Drought Monitor** and **gridMET** deficit.